# pSMAD — 00_conversion

**Feeds:** Fig 1e, 1f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 00 | CZI Conversion (Inline, No Wrapper Script)

This notebook converts raw `.czi` files to analysis-friendly formats, with all conversion logic visible inline.

For first-pass iteration, the defaults are safe:
- `RUN_CONVERSION = False`
- `DRY_RUN = True`

So you can inspect planned commands first, then enable real execution when ready.


## Cell Guide
1. Set conversion config (`RUN_CONVERSION`, `DRY_RUN`, tool path).
2. Review helper functions (`resolve_tool`, `build_command`).
3. Build the conversion plan table (no file writes yet).
4. Optionally execute conversion commands.
5. Inspect converted outputs and OME structure.


In [ ]:
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
import tifffile

In [ ]:
# -------------------------------
# User configuration
# -------------------------------
ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()

INPUT_DIR = ROOT / "data/well3-pSMAD-568pSMAD"
OUT_DIR = ROOT / "results/converted_tiff"
FORMAT = "ome-tiff"  # 'ome-tiff' or 'ome-zarr'

# Preferred tool location: keep Bio-Formats binaries inside this project.
TOOLS_DIR = ROOT / "tools/bftools"
PROJECT_BFCONVERT = TOOLS_DIR / "bfconvert"
PROJECT_BIOFORMATS2RAW = TOOLS_DIR / "bioformats2raw"

# TOOL_PATH selection policy:
# 1) use project-local tool if present
# 2) otherwise fall back to PATH lookup in resolve_tool(...)
if FORMAT == "ome-tiff" and PROJECT_BFCONVERT.exists():
    TOOL_PATH = PROJECT_BFCONVERT
elif FORMAT == "ome-zarr" and PROJECT_BIOFORMATS2RAW.exists():
    TOOL_PATH = PROJECT_BIOFORMATS2RAW
else:
    TOOL_PATH = None

OVERWRITE = True
RUN_CONVERSION = True
DRY_RUN = False
RESOLUTIONS = 4  # Used for ome-zarr mode.

print("Project root:", ROOT)
print("Input dir:", INPUT_DIR)
print("Output dir:", OUT_DIR)
print("Format:", FORMAT)
print("Project tools dir:", TOOLS_DIR)
print("Tool path override:", TOOL_PATH)

Project root: <analysis-root>/pSMAD
Input dir: <analysis-root>/pSMAD/data/well3-pSMAD-568pSMAD
Output dir: <analysis-root>/pSMAD/results/converted_tiff
Format: ome-tiff
Project tools dir: <analysis-root>/pSMAD/tools/bftools
Tool path override: <analysis-root>/pSMAD/tools/bftools/bfconvert


## Settings Explained

Use these flags as your safety controls:

- `RUN_CONVERSION`:
  - `False`: do **not** run conversion; only inspect plan/commands.
  - `True`: allow the notebook to iterate through files and issue conversion commands.
- `DRY_RUN` (only matters when `RUN_CONVERSION=True`):
  - `True`: print commands but do not execute.
  - `False`: execute conversion commands.
- `OVERWRITE`:
  - `False`: skip outputs that already exist.
  - `True`: replace existing converted outputs.
- `TOOL_PATH`:
  - auto-prefers project-local tools in `ROOT/tools/bftools`.
  - if no project-local binary exists, falls back to PATH lookup.
  - you can still set an explicit path manually if needed.

Recommended first-pass mode:
- `RUN_CONVERSION=False`
- `DRY_RUN=True`
- verify plan table
- then switch to `RUN_CONVERSION=True, DRY_RUN=False` when satisfied.

If project-local tools are missing and PATH lookup fails:
- place binaries in `ROOT/tools/bftools` (preferred), or
- set `TOOL_PATH` explicitly to any stable location.


In [ ]:
# -------------------------------
# Conversion helpers
# -------------------------------
def resolve_tool(format_name: str, explicit_tool: Path | None, dry_run: bool) -> str:
    """Resolve the converter executable path from explicit input or PATH."""
    if explicit_tool is not None:
        return str(explicit_tool)

    fallback = "bfconvert" if format_name == "ome-tiff" else "bioformats2raw"
    found = shutil.which(fallback)
    if found:
        return found
    if dry_run:
        return fallback
    raise RuntimeError(
        f"Required tool '{fallback}' not found on PATH. "
        "Install Bio-Formats CLI tools and retry, or set TOOL_PATH explicitly."
    )


def output_suffix(format_name: str) -> str:
    """Return the output filename suffix for the selected conversion format."""
    return ".ome.tif" if format_name == "ome-tiff" else ".ome.zarr"


def build_command(
    tool: str,
    src: Path,
    dst: Path,
    format_name: str,
    overwrite: bool,
    resolutions: int,
) -> list[str]:
    """Build the external conversion command for one source/destination pair."""
    if format_name == "ome-tiff":
        cmd = [tool]
        if overwrite:
            cmd.append("-overwrite")
        cmd.extend([str(src), str(dst)])
        return cmd

    cmd = [tool, str(src), str(dst), "--resolutions", str(resolutions)]
    if overwrite:
        cmd.append("--overwrite")
    return cmd

In [ ]:
# Build conversion plan table.
if not INPUT_DIR.exists():
    raise RuntimeError(f"Input directory not found: {INPUT_DIR}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
czi_files = sorted(INPUT_DIR.glob("*.czi"))
if not czi_files:
    raise RuntimeError(f"No .czi files found in {INPUT_DIR}")

suffix = output_suffix(FORMAT)
plan_rows = []
for src in czi_files:
    dst = OUT_DIR / f"{src.stem}{suffix}"
    plan_rows.append(
        {
            "source_file": src.name,
            "source_size_mb": round(src.stat().st_size / (1024**2), 2),
            "dest_file": dst.name,
            "dest_exists": dst.exists(),
            "will_run": OVERWRITE or (not dst.exists()),
        }
    )

plan_df = pd.DataFrame(plan_rows)
plan_df

                source_file  source_size_mb                     dest_file  \
0     well3-36locations.czi         1152.85     well3-36locations.ome.tif   
1       well3-568-pSMAD.czi         1501.10       well3-568-pSMAD.ome.tif   
2  well3-647-15BMPbeads.czi         1501.10  well3-647-15BMPbeads.ome.tif   
3              well3-BF.czi         1501.11              well3-BF.ome.tif   
4            well3-DAPI.czi         1501.10            well3-DAPI.ome.tif   

   dest_exists  will_run  
0         True      True  
1         True      True  
2         True      True  
3         True      True  
4         True      True  

In [ ]:
# Execute planned conversions (optional).
# Step 1: if RUN_CONVERSION is False, stop here (no tool lookup, no conversion).
# Step 2: if RUN_CONVERSION is True, resolve converter path.
# Step 3: build one CLI command per file and execute/print depending on DRY_RUN.
if not RUN_CONVERSION:
    print("RUN_CONVERSION=False -> no conversion executed.")
    print("Skipping tool resolution and conversion loop.")
else:
    tool = resolve_tool(FORMAT, TOOL_PATH, dry_run=DRY_RUN)
    print("Using tool:", tool)

    started = time.time()
    for src in czi_files:
        dst = OUT_DIR / f"{src.stem}{suffix}"
        if dst.exists() and not OVERWRITE:
            print(f"[SKIP] {dst} exists (set OVERWRITE=True to replace)")
            continue

        cmd = build_command(tool, src, dst, FORMAT, OVERWRITE, RESOLUTIONS)
        print("[CMD]", " ".join(cmd))

        if not DRY_RUN:
            subprocess.run(cmd, check=True)
            print(f"[OK] {src.name} -> {dst.name}")

    elapsed = time.time() - started
    print(f"Done in {elapsed:.1f}s")

Using tool: <analysis-root>/pSMAD/tools/bftools/bfconvert
[CMD] <analysis-root>/pSMAD/tools/bftools/bfconvert -overwrite <analysis-root>/pSMAD/data/well3-pSMAD-568pSMAD/well3-36locations.czi <analysis-root>/pSMAD/results/converted_tiff/well3-36locations.ome.tif
<analysis-root>/pSMAD/data/well3-pSMAD-568pSMAD/well3-36locations.czi
ZeissCZIReader initializing <analysis-root>/pSMAD/data/well3-pSMAD-568pSMAD/well3-36locations.czi
Unknown light source name 'Other Lamp'; assuming Laser
Unknown light source name 'Other Lamp'; assuming Laser
Unknown light source name 'Other Lamp'; assuming Laser
ome.xml.model.FilterSet@7bf3a5d8 reference to Dichroic:3 missing from object hierarchy.
ome.xml.model.FilterSet@6dd7b5a3 reference to Dichroic:1 missing from object hierarchy.
ome.xml.model.FilterSet@189cbd7c reference to Dichroic:2 missing from object hierarchy.
ome.xml.model.FilterSet@42e25b0b reference to Dichroic:4 missing from object hierarchy.
[Zeiss CZI] -> <analysis-root>/pSMAD/results/converte

In [ ]:
# Inspect converted outputs (quick structural check).
out_files = sorted(OUT_DIR.glob("*.ome.tif")) + sorted(OUT_DIR.glob("*.ome.zarr"))
if not out_files:
    print("No converted files found in output directory yet.")
else:
    rows = []
    for p in out_files:
        rec = {
            "file": p.name,
            "size_mb": round(p.stat().st_size / (1024**2), 2) if p.is_file() else None,
            "series_count": None,
            "series0_axes": None,
            "series0_shape": None,
        }
        if p.suffix.lower() in {".tif", ".tiff"}:
            with tifffile.TiffFile(p) as tif:
                rec["series_count"] = len(tif.series)
                if tif.series:
                    rec["series0_axes"] = tif.series[0].axes
                    rec["series0_shape"] = tuple(int(x) for x in tif.series[0].shape)
        rows.append(rec)

    df = pd.DataFrame(rows)
    display(df)

                           file  size_mb  series_count series0_axes  \
0     well3-36locations.ome.tif  1209.04            36          CYX   
1       well3-568-pSMAD.ome.tif  1440.24             5          ZYX   
2  well3-647-15BMPbeads.ome.tif  1440.23             5          ZYX   
3              well3-BF.ome.tif  1440.23             5          ZYX   
4            well3-DAPI.ome.tif  1440.23             5          ZYX   

       series0_shape  
0    (4, 2048, 2048)  
1  (3, 13005, 13005)  
2  (3, 13005, 13005)  
3  (3, 13005, 13005)  
4  (3, 13005, 13005)  